In [1]:
import pandas as pd
import numpy as np
import torch
import time
from tqdm import tqdm
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_metric
from comet import download_model, load_from_checkpoint
import matplotlib.pyplot as plt


/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load cleaned OpenSubtitles dataset
df = pd.read_csv("OpenSubtitles_en-fr_clean.csv")

# Optional: sample subset for faster testing
# df = df.sample(200, random_state=42).reset_index(drop=True)

# print(f"Loaded {len(df):,} sentence pairs.")
# print(df.head())

df = df.head(300).reset_index(drop=True)

print(f"Loaded first {len(df)} sentence pairs for evaluation.")
print(df.head())


Loaded 5,779 sentence pairs.
                                                 src  \
0                                       he has a gun   
1  oh, yeah? - i haven't lived yet. now i'm gonna...   
2                        hey, the camera, you got it   
3                   are you sure you want to do this   
4  the defendant is remanded to custody until the...   

                                                tgt src_lang tgt_lang  
0                                 il avait une arme       en       fr  
1           je n'ai pas vécu et déjà je dois mourir       en       fr  
2                         et la caméra, vous l'avez       en       fr  
3                    tu es sûre de vouloir faire ça       en       fr  
4  programmons une audience... la semaine prochaine       en       fr  


In [3]:
def split_into_contexts(sentence, n):
    """Split a sentence into chunks of n words."""
    words = sentence.strip().split()
    contexts = [" ".join(words[i:i+n]) for i in range(0, len(words), n)]
    return contexts


def translate_with_context(model, tokenizer, sentence, n, device="cpu"):
    """
    Translate a sentence by splitting into n-word chunks.
    Return the full translated sentence and total latency (sum of per-chunk times).
    """
    chunks = split_into_contexts(sentence, n)
    translated_chunks = []
    total_latency = 0.0

    for chunk in chunks:
        if not chunk.strip():
            continue
        start = time.time()
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True).to(device)
        outputs = model.generate(**inputs)
        end = time.time()

        translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_chunks.append(translated)
        total_latency += (end - start)

    return " ".join(translated_chunks), total_latency


In [4]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model loaded: {model_name}")


/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Model loaded: Helsinki-NLP/opus-mt-en-fr


In [ ]:
#context_sizes = [1, 2, 3, 5]
context_sizes = [1]
results = {}

for n in context_sizes:
    print(f"\nTranslating with context size = {n} ...")
    preds = []
    total_latencies = []

    for src in tqdm(df["src"], desc=f"context={n}"):
        pred, total_latency = translate_with_context(model, tokenizer, src, n, device=model.device)
        preds.append(pred)
        total_latencies.append(total_latency)

    results[n] = {
        "preds": preds,
        "latency": total_latencies
    }

    print(f"Mean total latency (sec/sentence): {np.mean(total_latencies):.3f}")



Translating with context size = 1 ...


context=1:   9%|▊         | 505/5779 [07:29<1:46:57,  1.22s/it]

In [ ]:
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

bleu_chrf_scores = []

for n, data in results.items():
    preds = data["preds"]
    bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in df["tgt"]])
    chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in df["tgt"]])
    bleu_chrf_scores.append({
        "context": n,
        "BLEU": bleu["score"],
        "chrF": chrf["score"]
    })
    print(f"\n===== Context = {n} =====")
    print(f"BLEU: {bleu['score']:.2f}")
    print(f"chrF: {chrf['score']:.2f}")


In [ ]:
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

comet_scores = []

for n, data in results.items():
    print(f"\nComputing COMET for context={n} ...")
    preds = data["preds"]
    data_batch = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(df["src"], preds, df["tgt"])]
    comet_score = comet_model.predict(
        data_batch,
        batch_size=4,
        gpus=0,
        num_workers=0,
        multiprocessing=False
    )
    comet_mean = np.mean(comet_score["system_score"])
    comet_scores.append({"context": n, "COMET": comet_mean})
    print(f"COMET: {comet_mean:.4f}")


In [ ]:
def compute_latency_metrics(tokenizer, src_texts, preds, total_latencies):
    """Compute AP, AL, DAL, and total latency per sentence."""
    src_lens = [len(tokenizer.encode(s)) for s in src_texts]
    tgt_lens = [len(tokenizer.encode(p)) for p in preds]

    AP = np.mean(np.array(tgt_lens) / np.array(src_lens))
    AL = 0.0  # offline system: no incremental decoding
    DAL = 0.0
    avg_total_latency = np.mean(total_latencies)

    return AP, AL, DAL, avg_total_latency


latency_scores = []
for n, data in results.items():
    preds = data["preds"]
    total_latencies = data["latency"]
    AP, AL, DAL, avg_latency = compute_latency_metrics(
        tokenizer, df["src"], preds, total_latencies
    )
    latency_scores.append({
        "context": n,
        "AP": AP,
        "AL": AL,
        "DAL": DAL,
        "TotalLatency(sec/sent)": avg_latency
    })


In [ ]:
df_bleu = pd.DataFrame(bleu_chrf_scores)
df_comet = pd.DataFrame(comet_scores)
df_latency = pd.DataFrame(latency_scores)

final_df = df_bleu.merge(df_comet, on="context").merge(df_latency, on="context")
final_df = final_df.sort_values("context").reset_index(drop=True)

print("\n===== Full Context Evaluation Summary =====")
print(final_df)



In [ ]:
plt.figure(figsize=(8,5))
plt.plot(final_df["context"], final_df["BLEU"], marker="o", label="BLEU")
plt.plot(final_df["context"], final_df["chrF"], marker="o", label="chrF")
plt.plot(final_df["context"], final_df["COMET"], marker="o", label="COMET")
plt.title("Translation Quality vs Context Size")
plt.xlabel("Context Size (n words)")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(final_df["context"], final_df["TotalLatency(sec/sent)"], marker="o", color="red")
plt.title("Latency vs Context Size")
plt.xlabel("Context Size (n words)")
plt.ylabel("Total Latency (sec/sentence)")
plt.grid(True)
plt.show()
